In [ ]:
# Cell 1: imports and project-root setup
import os
import sys
from pathlib import Path

def find_project_root(start: Path) -> Path:
    """Find the project root by searching upward for a directory that contains `src`."""
    start = start.resolve()
    candidates = [start] + list(start.parents)
    for candidate in candidates:
        if (candidate / "src").exists():
            return candidate
    return start

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import torch
from torch.utils.data import DataLoader
import gym
import d4rl  # noqa: F401

from src.experiment_config import *
from src.config import CHECKPOINTS_DIR, RAW_METRICS_DIR, OBS_STATS_DIR
from src.dataset import NoisyOfflineRLDataset
from src.encoder import DisentangledEncoder
from src.iql import IQLAgent
from src.train_eval import (
    eval_policy_on_env,
    load_and_freeze_encoder,
    train_iql_from_loader,
    save_metrics_json,
)


In [ ]:
def cross_covariance_loss(z1, z2):
    """
    Compute a cross-covariance penalty between two latent spaces.

    Both latent tensors are standardized across the batch dimension, and
    the full cross-covariance matrix is pushed toward zero. In this project,
    the objective is to reduce linear dependence between z_task and z_irrel.
    """
    # Standardize both latent spaces batch-wise.
    z1_norm = (z1 - z1.mean(dim=0)) / (z1.std(dim=0) + 1e-5)
    z2_norm = (z2 - z2.mean(dim=0)) / (z2.std(dim=0) + 1e-5)

    # Compute the cross-covariance matrix and penalize all entries.
    batch_size = z1.size(0)
    c = torch.mm(z1_norm.T, z2_norm) / batch_size
    loss = (c ** 2).mean()
    return loss


In [ ]:
# Cell 2: experiment configuration

METHOD = "disentangled_cov"

def noise_tag(noise_dim, noise_scale, noise_type):
    """Build a compact tag used in checkpoint and metrics directories."""
    ns = str(noise_scale).replace(".", "p")
    return f"nd{noise_dim}_ns{ns}_{noise_type}"

NOISE_TAG = noise_tag(NOISE_DIM, NOISE_SCALE, NOISE_TYPE)
SEED_TAG = f"seed_{SEED}"

# Save checkpoints, metrics, and observation statistics under the new project layout.
CKPT_DIR = CHECKPOINTS_DIR / METHOD / ENV_NAME / NOISE_TAG / SEED_TAG
METRICS_DIR = RAW_METRICS_DIR / METHOD / ENV_NAME / NOISE_TAG / SEED_TAG
OBS_DIR = OBS_STATS_DIR / METHOD / ENV_NAME / NOISE_TAG / SEED_TAG

CKPT_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)
OBS_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DEVICE:", DEVICE)
print("CKPT_DIR:", CKPT_DIR)
print("METRICS_DIR:", METRICS_DIR)
print("OBS_DIR:", OBS_DIR)


In [ ]:
# Cell 3: dataset and dataloader

dataset = NoisyOfflineRLDataset(
    env_name=ENV_NAME,
    noise_dim=NOISE_DIM,
    noise_scale=NOISE_SCALE,
    seed=SEED,
    use_timeouts=True,
    noise_type=NOISE_TYPE,
)

# Main dataloader used for IQL training.
train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
                          num_workers=4, pin_memory=True, persistent_workers=True)

# Infer dimensions directly from the dataset tensors.
state_dim = dataset.noisy_obs.shape[1]
action_dim = dataset.actions.shape[1]
true_state_dim = dataset.obs_dim

# Use the same heuristic latent-size rule as before.
LATENT_DIM = int(max(true_state_dim, NOISE_DIM) * 1.5)

np.savez(
    OBS_DIR / "obs_stats.npz",
    obs_mean=dataset.obs_mean,
    obs_std=dataset.obs_std,
    true_state_dim=true_state_dim,
)
print("Saved obs stats:", OBS_DIR / "obs_stats.npz")

print("state_dim (noisy):", state_dim)
print("true_state_dim:", true_state_dim)
print("action_dim:", action_dim)
print("latent_dim:", LATENT_DIM)


In [ ]:
# Cell 4: pretrain the disentangled encoder

torch.manual_seed(SEED)
np.random.seed(SEED)

encoder = DisentangledEncoder(
    state_dim=state_dim,
    action_dim=action_dim,
    true_state_dim=true_state_dim,
    latent_dim=LATENT_DIM,
).to(DEVICE)

optim = torch.optim.Adam(encoder.parameters(), lr=3e-4)

# Pretraining uses its own batch size from experiment_config.py unless overridden here.
pretrain_loader = DataLoader(dataset, batch_size=PRETRAIN_BS, shuffle=True, drop_last=True,
                             num_workers=4, pin_memory=True, persistent_workers=True)

for epoch in range(1, PRETRAIN_EPOCHS + 1):
    encoder.train()
    losses = []

    # Dataset format:
    # (obs, act, next_obs, rew, done, pure_obs, pure_next_obs)
    for obs, act, next_obs, rew, done, pure_obs, pure_next_obs in pretrain_loader:
        obs = obs.to(DEVICE)
        act = act.to(DEVICE)
        next_obs = next_obs.to(DEVICE)
        rew = rew.to(DEVICE)
        done = done.to(DEVICE)
        pure_obs = pure_obs.to(DEVICE)
        pure_next_obs = pure_next_obs.to(DEVICE)

        # Encode noisy observations into task and nuisance latents.
        z_task, z_irrel = encoder(obs)

        # Predict the next clean state from (z_task, action).
        pred_next_true = encoder.state_predictor(torch.cat([z_task, act], dim=-1))
        loss_dyn = torch.nn.functional.mse_loss(pred_next_true, pure_next_obs)

        # Predict reward from the task-relevant latent.
        pred_rew = encoder.reward_predictor(z_task)
        loss_rew = torch.nn.functional.mse_loss(pred_rew.squeeze(-1), rew)

        # Penalize cross-covariance between task and nuisance latents.
        loss_ind = cross_covariance_loss(z_task, z_irrel)

        # The cross-covariance penalty is averaged, so a larger coefficient is used here.
        indep_weight = 50.0
        if epoch == 1 and len(losses) == 0:
            print(
                f"[{METHOD} debug] dyn={loss_dyn.item():.4f}, "
                f"weighted_indep={indep_weight * loss_ind.item():.4f}"
            )

        loss = loss_dyn + loss_rew + indep_weight * loss_ind

        optim.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(encoder.parameters(), max_norm=1.0)
        optim.step()

        losses.append(loss.detach())

    print(f"[pretrain] epoch {epoch}, loss={torch.stack(losses).mean().item():.4f}")

# Save the final encoder checkpoint for later IQL training.
CKPT_ENCODER = CKPT_DIR / f"encoder_epoch_{PRETRAIN_EPOCHS}.pth"
torch.save(encoder.state_dict(), CKPT_ENCODER)
print("Saved encoder:", CKPT_ENCODER)


In [ ]:
# Cell 5: load and freeze the encoder, then train IQL

encoder = load_and_freeze_encoder(
    encoder=encoder,
    ckpt_path=CKPT_ENCODER,
    device=DEVICE,
)

iql = IQLAgent(
    latent_dim=LATENT_DIM,
    action_dim=action_dim,
    device=DEVICE,
    expectile=0.7,
    temperature=3.0,
    discount=0.99,
)

iql_history = train_iql_from_loader(
    iql=iql,
    train_loader=train_loader,
    device=DEVICE,
    epochs=EPOCHS,
    ckpt_dir=CKPT_DIR,
    method=METHOD,
    save_every=10,
    encoder=encoder,
    repr_mode="disentangled",
    use_tqdm=False,
)


In [ ]:
# Cell 6: evaluate and save metrics

print("Start evaluating ...")
metrics = eval_policy_on_env(
    iql=iql,
    env_name=ENV_NAME,
    encoder=encoder,
    method=METHOD,
    obs_mean=dataset.obs_mean,
    obs_std=dataset.obs_std,
    true_state_dim=true_state_dim,
    noise_dim=NOISE_DIM,
    noise_scale=NOISE_SCALE,
    noise_type=NOISE_TYPE,
    episodes=20,
    max_steps=1000,
    seed=SEED,
    device=DEVICE,
    use_fixed_noise=True,
)

metrics_path = save_metrics_json(
    metrics_dir=METRICS_DIR,
    metrics=metrics,
    env_name=ENV_NAME,
    method=METHOD,
    seed=SEED,
    noise_dim=NOISE_DIM,
    noise_scale=NOISE_SCALE,
    noise_type=NOISE_TYPE,
    extra={
        "latent_dim": LATENT_DIM,
        "pretrain_epochs": PRETRAIN_EPOCHS,
        "iql_epochs": EPOCHS,
        "encoder_checkpoint": str(CKPT_ENCODER),
        "obs_stats_path": str(OBS_DIR / "obs_stats.npz"),
        "iql_history": iql_history,
    },
)

print("Saved metrics:", metrics_path)
print(metrics)
